In [1]:
from Module01_AgenticRAG.ingest import load_faq_data

In [2]:
documents = load_faq_data()

In [3]:
documents[600]

{'id': '1e829a8c6f',
 'course': 'llm-zoomcamp',
 'section': 'Module 2: Vector Search',
 'question': 'Do I need a new GitHub repo for Module 2, or just a new codespace?',
 'answer': "Just a new codespace. A codespace is an environment (see *Can I run the course locally instead of Codespaces?*); you create it from your existing repository, so you don't need a new GitHub repo.\n\nUse a separate codespace for Module 2 because the vector-search dependencies are fairly heavy. Keeping them isolated means you can simply stop or delete that codespace when you're done, rather than leaving the extra weight in your Module 1 environment. Setup is quick.\n\nSince you may delete this codespace later, commit your work to your repository (see *What happens to code saved in Codespaces if I do not commit it?*)."}

In [4]:
documents_llm = []

for doc in documents:
    if doc['course'] == 'llm-zoomcamp':
        documents_llm.append(doc)
len(documents_llm)

115

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [34]:
from pydantic import BaseModel


class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [9]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("API_URL"),
)

In [10]:
import json

user_prompt = json.dumps(doc)
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
response = openai_client.responses.parse(
    model='openai/gpt-oss-20b',
    input=messages,
    text_format=Questions
)

In [13]:
results = response.output_parsed.questions

In [14]:
results

["Is it still possible to enroll in this course now that I've just found it?",
 'Can I join the LLM Zoomcamp even if I discover it just today?',
 'I only just saw this course, can I register at this stage?',
 'Will I receive a certificate if I start the course late?',
 'What is the deadline for project submission to get a certificate?']

In [15]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    client=openai_client,
    instructions=data_gen_instructions,
    user_prompt=user_prompt,
    model='meta-llama/llama-4-scout-17b-16e-instruct',
    output_type=Questions
)


In [16]:
result, usage

(Questions(questions=['What are the course prerequisites?', 'How do I submit my project?', 'What is the deadline for submission?']),
 ResponseUsage(input_tokens=170, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=36, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=206))

In [17]:
from evaluation_utils import llm_structured_retry


def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    output, usage = llm_structured_retry(
        client=openai_client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        model='meta-llama/llama-4-scout-17b-16e-instruct',
        output_type=Questions
    )

    resuls = []

    for q in output.questions:
        resuls.append(
            {
                "question": q,
                "course": doc['course'],
                "document": doc["id"]
            }
        )
    return results, usage

In [31]:
documents[1]

{'id': '977bf7786c',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."}

In [ ]:
from tqdm.auto import tqdm

ground_truth = {
    'question': [],
    'document': []
}
usages = []
failed_docs = []

for doc in tqdm(documents):
        records, usage = generate_ground_truth(doc)
        ground_truth['question'].extend(records)

        ground_truth['document'].extend([doc['id']] * len(records))
        usages.append(usage)

  0%|          | 0/115 [00:00<?, ?it/s]

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=8) as pool:
    results = map_progress(pool, documents[:len(documents) // 2], generate_ground_truth)


  0%|          | 0/57 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-16e-instruct` in organization `org_01kvw2kpzsfrv8gaagb9g013gg` service tier `on_demand` on requests per minute (RPM): Limit 30, Used 30, Requested 1. Please try again in 2s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'requests', 'code': 'rate_limit_exceeded'}}

In [ ]:
with ThreadPoolExecutor(max_workers=8) as pool:
    results2 = map_progress(pool, documents[len(documents) // 2:], generate_ground_truth)

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)